# Model 1 — Basic Seq2Seq (LSTM, no attention)

## Part 1: Data pre-processing

Steps covered in this section:
1. Load and filter WikiQA splits
2. Tokenise questions and answers
3. Build vocabulary from training data
4. Encode sequences to integer IDs
5. Create PyTorch Datasets and DataLoaders

In [1]:
import re
from collections import Counter

import nltk
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

nltk.download("punkt_tab", quiet=True)

DATA_DIR = "../data/raw"

### 1. Load and filter WikiQA splits

Each TSV row is a (question, candidate sentence, label) triple. We keep only `Label=1` rows, which are the correct Q&A pairs.

In [2]:
def load_split(filename):
    df = pd.read_csv(f"{DATA_DIR}/{filename}", sep="\t")
    df = df[df["Label"] == 1][["Question", "Sentence"]].reset_index(drop=True)
    return df

train_df = load_split("WikiQA-train.tsv")
dev_df   = load_split("WikiQA-dev.tsv")
test_df  = load_split("WikiQA-test.tsv")

print(f"Q&A pairs — train: {len(train_df):,}  dev: {len(dev_df):,}  test: {len(test_df):,}")
train_df.head(3)

Q&A pairs — train: 1,039  dev: 140  test: 291


,Question,Sentence
0,how are glacier caves formed?,A glacier cave is a cave formed within the ice...
1,how much is 1 tablespoon of water,This tablespoon has a capacity of about 15 mL.
2,how much is 1 tablespoon of water,In the USA one tablespoon (measurement unit) i...


### 2. Tokenisation

Lowercase, strip non-alphanumeric characters (keeping apostrophes and `?`), then use NLTK's word tokeniser.

In [3]:
def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9'? ]", " ", text)
    return nltk.word_tokenize(text)

# Sanity check
print(tokenize("How are glacier caves formed?"))
print(tokenize("A glacier cave is a cave formed within the ice of a glacier."))

['how', 'are', 'glacier', 'caves', 'formed', '?']
['a', 'glacier', 'cave', 'is', 'a', 'cave', 'formed', 'within', 'the', 'ice', 'of', 'a', 'glacier']


In [4]:
def tokenize_df(df):
    df = df.copy()
    df["q_tokens"] = df["Question"].apply(tokenize)
    df["a_tokens"] = df["Sentence"].apply(tokenize)
    return df

train_df = tokenize_df(train_df)
dev_df   = tokenize_df(dev_df)
test_df  = tokenize_df(test_df)

train_df[["Question", "q_tokens", "Sentence", "a_tokens"]].head(3)

,Question,q_tokens,Sentence,a_tokens
0,how are glacier caves formed?,"[how, are, glacier, caves, formed, ?]",A glacier cave is a cave formed within the ice...,"[a, glacier, cave, is, a, cave, formed, within..."
1,how much is 1 tablespoon of water,"[how, much, is, 1, tablespoon, of, water]",This tablespoon has a capacity of about 15 mL.,"[this, tablespoon, has, a, capacity, of, about..."
2,how much is 1 tablespoon of water,"[how, much, is, 1, tablespoon, of, water]",In the USA one tablespoon (measurement unit) i...,"[in, the, usa, one, tablespoon, measurement, u..."


### Pre-processing analysis

Before building the vocabulary, check for issues that would hurt model quality.

In [5]:

a_lens = train_df["a_tokens"].str.len()
q_lens = train_df["q_tokens"].str.len()

print("Answer length percentiles (train):")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p:3d}: {a_lens.quantile(p/100):.0f} tokens")

print(f"\nLongest answer ({a_lens.max()} tokens):")
worst = train_df.loc[a_lens.idxmax()]
print(f"  Q: {worst['Question']}")
print(f"  A: {worst['Sentence'][:120]}...")

print("\nUNK rate on dev with different MIN_FREQ values:")
all_train = train_df["q_tokens"].tolist() + train_df["a_tokens"].tolist()
train_counts = Counter(tok for toks in all_train for tok in toks)
dev_all_tokens = [t for toks in dev_df["q_tokens"].tolist() + dev_df["a_tokens"].tolist() for t in toks]
for mf in [1, 2, 3]:
    known = {t for t, c in train_counts.items() if c >= mf}
    unk_rate = sum(1 for t in dev_all_tokens if t not in known) / len(dev_all_tokens) * 100
    vocab_size = len(known) + 4
    print(f"  MIN_FREQ={mf}: vocab size={vocab_size:,}  dev UNK rate={unk_rate:.1f}%")

Answer length percentiles (train):
  p 50: 24 tokens
  p 75: 33 tokens
  p 90: 42 tokens
  p 95: 47 tokens
  p 99: 60 tokens
  p100: 165 tokens

Longest answer (165 tokens):
  Q: where is keith whitley from
  A: Jackie Keith Whitley (July 1, 1954Stambler, Irwin, and Grelun Landon (2000). - Country Music: The Encyclopedia. - New Yo...

UNK rate on dev with different MIN_FREQ values:
  MIN_FREQ=1: vocab size=7,272  dev UNK rate=17.9%
  MIN_FREQ=2: vocab size=3,380  dev UNK rate=23.9%
  MIN_FREQ=3: vocab size=2,084  dev UNK rate=29.0%


### 3. Build vocabulary

Built from **training data only** to avoid leakage. `MIN_FREQ=1` keeps all training tokens — the dataset is small (~1,000 pairs) so discarding rare words with `MIN_FREQ=2` raises the dev UNK rate from 17.9% to 23.9% without a meaningful vocab-size benefit.

Special tokens:
| Token | ID |
| --- | --- |
| `<pad>` | 0 |
| `<sos>` | 1 |
| `<eos>` | 2 |
| `<unk>` | 3 |

In [6]:
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
MIN_FREQ = 1
MAX_ANS_LEN = 60  # p99 of training answers is 60 tokens; caps the one malformed 165-token outlier

class Vocabulary:
    def __init__(self):
        self.token2idx = {"<pad>": PAD_IDX, "<sos>": SOS_IDX, "<eos>": EOS_IDX, "<unk>": UNK_IDX}
        self.idx2token = {v: k for k, v in self.token2idx.items()}

    def build(self, token_lists: list[list[str]]) -> None:
        counts = Counter(tok for tokens in token_lists for tok in tokens)
        for token, freq in counts.items():
            if freq >= MIN_FREQ and token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token

    def encode(self, tokens: list[str]) -> list[int]:
        return [SOS_IDX] + [self.token2idx.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]

    def decode(self, ids: list[int]) -> list[str]:
        skip = {PAD_IDX, SOS_IDX, EOS_IDX}
        return [self.idx2token.get(i, "<unk>") for i in ids if i not in skip]

    def __len__(self):
        return len(self.token2idx)

all_train_tokens = train_df["q_tokens"].tolist() + train_df["a_tokens"].tolist()
vocab = Vocabulary()
vocab.build(all_train_tokens)

print(f"Vocabulary size: {len(vocab):,} (including 4 special tokens)")

Vocabulary size: 7,272 (including 4 special tokens)


### 4. Encode sequences

Each token list is wrapped with `<sos>`/`<eos>` and mapped to integer IDs. Answers are truncated to `MAX_ANS_LEN` before encoding to cap the one malformed 165-token outlier and keep training stable.

In [7]:
def encode_df(df, vocab):
    df = df.copy()
    df["q_ids"] = df["q_tokens"].apply(vocab.encode)
    df["a_ids"] = df["a_tokens"].apply(lambda toks: vocab.encode(toks[:MAX_ANS_LEN]))
    return df

train_df = encode_df(train_df, vocab)
dev_df   = encode_df(dev_df, vocab)
test_df  = encode_df(test_df, vocab)

# Sanity check — encode then decode should recover the original tokens
sample = train_df.iloc[0]
print("Original Q tokens :", sample["q_tokens"])
print("Encoded            :", sample["q_ids"])
print("Decoded            :", vocab.decode(sample["q_ids"]))

# Confirm the outlier is capped
max_a = max(len(ids) for ids in train_df["a_ids"])
print(f"\nMax encoded answer length after truncation: {max_a} (was 165)")

Original Q tokens : ['how', 'are', 'glacier', 'caves', 'formed', '?']
Encoded            : [1, 4, 5, 6, 7, 8, 9, 2]
Decoded            : ['how', 'are', 'glacier', 'caves', 'formed', '?']

Max encoded answer length after truncation: 62 (was 165)


### 5. PyTorch Dataset and DataLoaders

Sequences are padded to the longest sequence in each batch (dynamic padding), keeping memory use low.

In [8]:
BATCH_SIZE = 32

class WikiQADataset(Dataset):
    def __init__(self, df):
        self.q_ids = df["q_ids"].tolist()
        self.a_ids = df["a_ids"].tolist()

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.q_ids[idx]), torch.tensor(self.a_ids[idx])


def collate_fn(batch):
    questions, answers = zip(*batch)
    src = pad_sequence(questions, batch_first=True, padding_value=PAD_IDX)
    tgt = pad_sequence(answers,   batch_first=True, padding_value=PAD_IDX)
    src_lens = torch.tensor([len(q) for q in questions])
    return {"src": src, "tgt": tgt, "src_lens": src_lens}


train_loader = DataLoader(WikiQADataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(WikiQADataset(dev_df),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(WikiQADataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Batches — train: {len(train_loader)}  dev: {len(dev_loader)}  test: {len(test_loader)}")

batch = next(iter(train_loader))
print("\nSample batch:")
print(f"  src shape : {batch['src'].shape}")
print(f"  tgt shape : {batch['tgt'].shape}")
print(f"  src_lens  : {batch['src_lens'][:5].tolist()}")

Batches — train: 33  dev: 5  test: 10

Sample batch:
  src shape : torch.Size([32, 15])
  tgt shape : torch.Size([32, 62])
  src_lens  : [7, 9, 8, 8, 11]
